In [6]:
from tensorflow import keras
from sklearn.model_selection import train_test_split

In [7]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

4422102/4422102 [==============================] - 0s 0us/step


# Data preprocessing

- Scale pixel values ​​between 0 and 1
- Convert 2-dimensional array to 1-dimensional array
- Training and verification divided

In [8]:
scaled_train = x_train / 255
scaled_train = scaled_train.reshape(-1, 28 * 28)
scaled_train, scaled_val, y_train, y_val = train_test_split(scaled_train, y_train,
                                                            test_size = 0.2,
                                                            stratify = y_train,
                                                            random_state = 14)

# Deep Neural Network
- The difference from a single-layer neural network is that a dense layer is added between the input layer and the output layer.
    - All layers between the input layer and the output layer are called hidden layers.
    
- There is a difference between the activation function applied to the output layer and the activation function applied to the hidden layer.
    - Activation function of output layer
        - The role is to guide results to be output in an appropriate format so that they can be compared well with the data set.
        - There are limitations in types (binary classification: sigmoid, multiple classification: softmax)
        
    - Activation function of hidden layer
        - Used between multiple layers
        - Freedom of choice compared to output layer functions
        - Most common activation function: ReLU
        - The hidden layer of every neural network always has an activation function.

- Activation function
    - Reasons for using activation functions
        - Example) a x 4 + 2 = b
        - b x 3 - 5 = c
        - The above two equations can be simplified to a x 12 + 1 = c.
        
    - If the hidden layer only performs linear arithmetic calculations, even if the layer becomes deeper, the function is simplified and learning efficiency decreases.
        - Therefore, a process of twisting linear calculations into non-linear calculations using the activation function is necessary.

In [9]:
# hidden layer
dense1 = keras.layers.Dense(100, activation="sigmoid", input_shape=(784,))

In [10]:
# output layer
dense2 = keras.layers.Dense(10, activation="softmax")

- dense1
    - Hidden layer
    - Dense layer with 100 units
        - There is no special standard for determining the number of units.
        - However, it must be more than the number of units in the output layer.
            - If the hidden layer has fewer units than the output layer, the amount of information transmitted may be insufficient.
    - Activation function is sigmoid
    - Because it is connected to the input layer, the size of the input is (784,)
    
- dense2
    - Classified into 10 classes, so 10 units
    - Because it is multiple classification, the activation function is softmax.

In [11]:
model = keras.Sequential()

- **Must be added in order from the first hidden layer to the last output layer**


In [12]:
model.add(dense1)
model.add(dense2)

In [13]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 100)               78500     
                                                                 
 dense_1 (Dense)             (None, 10)                1010      
                                                                 
Total params: 79510 (310.59 KB)
Trainable params: 79510 (310.59 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


- Model summary information
    - Layers included in the model are listed in order
        - From the first hidden layer to the output layer
    - The name, class, output size, and number of parameters are displayed for each layer.
        - name
            - Can be specified as name parameter when creating a layer
            - If not specified, defaults to "dense"
            
    -Output Shape
        - Output size
        - (None, 100)
            - The first dimension refers to the number of samples
            - The reason the number of samples is None is because it is unknown how many images will be used at once, so it is set to None to flexibly respond to any batch size.
                - Keras basically uses mini-batch gradient descent.
                - If batch_size is not set, default value is 32.
                - Therefore, the first dimension of input_shape or output_shape is also called batch dimension.
                
            - The second dimension is the number of outputs
                - Since the results come from 100 units, the number of outputs is 100.
                - In other words, 784 pixel values ​​for each image pass through the hidden layer and are compressed into 100 features.
    - Param
        - Number of model parameters
        - dense layer
            - Weights for all combinations of 784 pixel input values ​​and 100 units + 1 intercept for each unit
                - 784 * 100 + 100 = 78500
                
        - dense_1 layer
            - Weights for all combinations of the 100 units in the front hidden layer and the 10 output layer units + 1 intercept for each unit
                - 100 * 10 + 10 = 1010

In [14]:
model = keras.Sequential([
    keras.layers.Dense(100, activation = "sigmoid", input_shape = (784,), name = "hidden"),
    keras.layers.Dense(10, activation = "softmax", name = "output")
], name = "Fashion_MNIST_model")

In [15]:
model.summary()

Model: "Fashion_MNIST_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 hidden (Dense)              (None, 100)               78500     
                                                                 
 output (Dense)              (None, 10)                1010      
                                                                 
Total params: 79510 (310.59 KB)
Trainable params: 79510 (310.59 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [16]:
model.compile(loss = "sparse_categorical_crossentropy", metrics = ["accuracy"])
model.fit(scaled_train, y_train, epochs = 5, batch_size = 32)

Epoch 1/5
1500/1500 [==============================] - 8s 5ms/step - loss: 0.5741 - accuracy: 0.8047
Epoch 2/5
1500/1500 [==============================] - 5s 4ms/step - loss: 0.4104 - accuracy: 0.8519
Epoch 3/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.3768 - accuracy: 0.8639
Epoch 4/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.3542 - accuracy: 0.8718
Epoch 5/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.3381 - accuracy: 0.8767


- Performance on the training set is improved with the addition of hidden layers.

# Activation Function

- The activation function often used in the hidden layer of early artificial neural networks is sigmoid.


- No matter how large the input value is, it is output as a value between 0 and 1, so the range of the output value is too narrow.
    - When performing gradient descent, gradient vanishing can occur, where the gradient converges to 0.
    - As the number of layers increases and the model becomes more complex, the effects accumulate, making learning more difficult.

## ReLU Function

- If the input is a positive number, it is passed through as if there is no activation function. If the input is a negative number, it becomes 0.

- Expression: max(0, z)

# Flatten

- Until now, because the size of the fashion MNIST data was 28 * 28, it was spread in one dimension using reshape before injecting it into the artificial neural network.
- For the same function, Keras provides a Flatten layer.
    - Role of unfolding all remaining input dimensions in a row except for the number of samples dimension
    - No weights or intercepts
    - However, since it is added between the input layer and the hidden layer, it is called a layer for convenience, but it is not considered to have increased the depth of the neural network.

In [17]:
model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape = (28, 28)))
model.add(keras.layers.Dense(100, activation = "relu"))
model.add(keras.layers.Dense(10, activation = "softmax"))

In [18]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense_2 (Dense)             (None, 100)               78500     
                                                                 
 dense_3 (Dense)             (None, 10)                1010      
                                                                 
Total params: 79510 (310.59 KB)
Trainable params: 79510 (310.59 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


- Parameter of Flatten layer is 0
- The advantage of adding a Flatten layer is that you can guess the dimension of the input value.
    - It is clearly revealed that 784 inputs are passed to the first hidden layer.
- One of the philosophies of the Keras API is to include preprocessing of input data in the model as much as possible.

# Re-prepare the data for a new model

In [19]:
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
scaled_train = x_train / 255
scaled_train, scaled_val, y_train, y_val = train_test_split(scaled_train, y_train,
                                                            test_size = 0.2,
                                                            stratify = y_train,
                                                            random_state = 14)

In [20]:
scaled_train.shape

(48000, 28, 28)

In [22]:
model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(scaled_train, y_train, epochs=5)

Epoch 1/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.5404 - accuracy: 0.8087
Epoch 2/5
1500/1500 [==============================] - 7s 5ms/step - loss: 0.3999 - accuracy: 0.8562
Epoch 3/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.3606 - accuracy: 0.8706
Epoch 4/5
1500/1500 [==============================] - 7s 5ms/step - loss: 0.3384 - accuracy: 0.8785
Epoch 5/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.3214 - accuracy: 0.8856


In [23]:
model.evaluate(scaled_val, y_val)

375/375 [==============================] - 1s 2ms/step - loss: 0.3302 - accuracy: 0.8863


[0.33021312952041626, 0.8863333463668823]

# Deep learning hyperparameters

- Hyperparameters: Parameters that must be specified by a person because the model doesn't learn them

- Types of hyperparameters in artificial neural networks
    - Number of hidden layers
    - Number of units in hidden layer
    - Activation function
    - Types of floors
        - Dense layer, CNN, RNN
    - Number of mini-batches (batch_size)
    - Number of repetitions (epochs)
    - Optimizer
    - Learning rate of the optimizer

## Optimizer

- Keras default optimizer is set to gradient descent algorithm (RMSprop).
- In addition, various gradient descent algorithms are provided, and these are called optimizers.

In [26]:
# use sgd optimizer
sgd = keras.optimizers.SGD()
model.compile(optimizer=sgd, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
#OR
model.compile(optimizer = "sgd", loss = "sparse_categorical_crossentropy",
              metrics = ["accuracy"])

In [27]:
# If you want to adjust the learning rate of the optimizer
sgd = keras.optimizers.SGD(learning_rate=0.1)

# Types of Optimizers
- Momentum
    - The default value of momentum in the SGD class is 0.
    - If momentum is specified as a value greater than 0, momentum optimization is used.
    - Generally, the momentum parameter is set to 0.9 or higher.
    
- NAG (Nesterov Accelerated Gradient)
    - Nesterov momentum optimization can be used by changing the nesterov parameter of the SGD class from the default value of False to True.
    - In most cases, Nesterov momentum optimization provides better performance than basic gradient descent.

In [28]:
nag = keras.optimizers.SGD(momentum=0.9, nesterov=True)

- adaptive learning rate
    - As the model approaches the optimal point, the learning rate is lowered.
        - High possibility of stably converging to the optimal point

In [29]:
# Adagrad
adagrad = keras.optimizers.Adagrad()

In [30]:
# RMSProp
rmsprop = keras.optimizers.RMSprop()

In [31]:
model.compile(optimizer=rmsprop, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

- Adam is a combination of the above momentum optimization and adaptive learning rate.
    - most used optimizer

In [32]:
model = keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(28, 28)))
model.add(keras.layers.Dense(100, activation="relu"))
model.add(keras.layers.Dense(10, activation="softmax"))

In [34]:
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [38]:
model.fit(scaled_train, y_train, epochs=5)

Epoch 1/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.4066 - accuracy: 0.8536
Epoch 2/5
1500/1500 [==============================] - 9s 6ms/step - loss: 0.3594 - accuracy: 0.8687
Epoch 3/5
1500/1500 [==============================] - 5s 4ms/step - loss: 0.3292 - accuracy: 0.8804
Epoch 4/5
1500/1500 [==============================] - 7s 5ms/step - loss: 0.3123 - accuracy: 0.8844
Epoch 5/5
1500/1500 [==============================] - 6s 4ms/step - loss: 0.2957 - accuracy: 0.8919


In [39]:
model.evaluate(scaled_val, y_val)

375/375 [==============================] - 1s 3ms/step - loss: 0.3111 - accuracy: 0.8864


[0.3110579252243042, 0.8864166736602783]